In [1]:
# ============================================================
# FragDenStaat: ALLE Dokument-Metadaten herunterladen
# ============================================================
#
# Dieses Skript:
# - lädt automatisch alle API-Seiten
# - speichert jedes Dokument sofort in einer JSONL-Datei
# - merkt sich die nächste Seite in einer Checkpoint-Datei
# - kann nach einem Abbruch fortgesetzt werden
# - versucht Verbindungsfehler automatisch erneut
# - erzeugt am Ende zusätzlich eine CSV-Datei
#
# Es lädt noch KEINE PDF-Dateien herunter.
# ============================================================

import json
import time
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# ============================================================
# 1. EINSTELLUNGEN
# ============================================================

START_URL = "https://fragdenstaat.de/api/v1/document/?limit=50&offset=0"

# Pause zwischen zwei API-Seiten.
# Nicht auf 0 setzen, damit der Server nicht unnötig belastet wird.
PAUSE_SECONDS = 1.0

# Nach jeweils so vielen Seiten wird ein Fortschrittsbericht ausgegeben.
REPORT_EVERY_PAGES = 10

# None bedeutet: alle Seiten laden.
# Zum Testen kannst du vorübergehend 5 eintragen.
MAX_PAGES = None

# Arbeitsordner
BASE_DIR = Path.cwd()

# Ausgabedateien
JSONL_FILE = BASE_DIR / "fragdenstaat_alle_dokumente.jsonl"
CSV_FILE = BASE_DIR / "fragdenstaat_alle_dokumente.csv"
CHECKPOINT_FILE = BASE_DIR / "fragdenstaat_checkpoint.json"
ERROR_LOG_FILE = BASE_DIR / "fragdenstaat_fehler.log"


# ============================================================
# 2. HTTP-SESSION MIT AUTOMATISCHEN WIEDERHOLUNGEN
# ============================================================

retry_strategy = Retry(
    total=8,
    connect=8,
    read=8,
    status=8,
    backoff_factor=2,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
    raise_on_status=False,
)

session = requests.Session()

session.headers.update(
    {
        "Accept": "application/json",
        "User-Agent": (
            "OpenLens-Capstone/0.1 "
            "(educational metadata research; polite pagination)"
        ),
    }
)

adapter = HTTPAdapter(
    max_retries=retry_strategy,
    pool_connections=1,
    pool_maxsize=1,
)

session.mount("https://", adapter)
session.mount("http://", adapter)


# ============================================================
# 3. HILFSFUNKTIONEN
# ============================================================

def load_checkpoint() -> dict[str, Any]:
    """
    Lädt den letzten Speicherstand.

    Falls keine Checkpoint-Datei existiert, beginnt der Download
    bei der ersten API-Seite.
    """

    if not CHECKPOINT_FILE.exists():
        return {
            "next_url": START_URL,
            "pages_downloaded": 0,
            "documents_saved": 0,
            "finished": False,
        }

    try:
        with CHECKPOINT_FILE.open("r", encoding="utf-8") as file:
            checkpoint = json.load(file)

        if not isinstance(checkpoint, dict):
            raise ValueError("Checkpoint ist kein Dictionary.")

        return checkpoint

    except (json.JSONDecodeError, OSError, ValueError) as error:
        raise RuntimeError(
            f"Checkpoint konnte nicht gelesen werden:\n"
            f"{CHECKPOINT_FILE}\n\n"
            f"Fehler: {error}"
        ) from error


def save_checkpoint(
    next_url: str | None,
    pages_downloaded: int,
    documents_saved: int,
    finished: bool,
) -> None:
    """
    Speichert den aktuellen Fortschritt.
    """

    checkpoint = {
        "next_url": next_url,
        "pages_downloaded": pages_downloaded,
        "documents_saved": documents_saved,
        "finished": finished,
        "saved_at": pd.Timestamp.now(tz="UTC").isoformat(),
    }

    temporary_file = CHECKPOINT_FILE.with_suffix(".tmp")

    with temporary_file.open("w", encoding="utf-8") as file:
        json.dump(
            checkpoint,
            file,
            ensure_ascii=False,
            indent=2,
        )

    temporary_file.replace(CHECKPOINT_FILE)


def log_error(message: str) -> None:
    """
    Schreibt Fehlermeldungen zusätzlich in eine Log-Datei.
    """

    timestamp = pd.Timestamp.now(tz="UTC").isoformat()

    with ERROR_LOG_FILE.open("a", encoding="utf-8") as file:
        file.write(f"[{timestamp}] {message}\n")


def request_json(url: str) -> dict[str, Any]:
    """
    Ruft eine API-Seite ab und gibt die JSON-Antwort zurück.
    """

    response = session.get(
        url,
        timeout=(15, 120),
    )

    response.raise_for_status()

    try:
        data = response.json()
    except requests.exceptions.JSONDecodeError as error:
        preview = response.text[:500]

        raise ValueError(
            "Die API-Antwort war kein gültiges JSON.\n"
            f"Antwortanfang:\n{preview}"
        ) from error

    if not isinstance(data, dict):
        raise TypeError(
            "Die API-Antwort muss ein JSON-Objekt sein."
        )

    if "objects" not in data:
        raise KeyError(
            'In der API-Antwort fehlt der Schlüssel "objects".'
        )

    if not isinstance(data["objects"], list):
        raise TypeError(
            'Der Wert unter "objects" ist keine Liste.'
        )

    return data


def write_documents(
    documents: list[dict[str, Any]],
    output_file,
) -> int:
    """
    Schreibt Dokumente zeilenweise in die JSONL-Datei.
    """

    saved = 0

    for document in documents:
        output_file.write(
            json.dumps(
                document,
                ensure_ascii=False,
            )
            + "\n"
        )
        saved += 1

    output_file.flush()

    return saved


# ============================================================
# 4. CHECKPOINT LADEN
# ============================================================

checkpoint = load_checkpoint()

next_url = checkpoint.get("next_url", START_URL)
pages_downloaded = int(checkpoint.get("pages_downloaded", 0))
documents_saved = int(checkpoint.get("documents_saved", 0))
already_finished = bool(checkpoint.get("finished", False))


print("=" * 65)
print("FRAGDENSTAAT-METADATEN-DOWNLOAD")
print("=" * 65)
print("Arbeitsordner:       ", BASE_DIR)
print("JSONL-Datei:         ", JSONL_FILE)
print("CSV-Datei:           ", CSV_FILE)
print("Checkpoint-Datei:    ", CHECKPOINT_FILE)
print("Bereits geladene Seiten:", pages_downloaded)
print("Bereits gespeicherte Dokumente:", documents_saved)
print()


if already_finished:
    print("Der Checkpoint meldet, dass der Download bereits abgeschlossen ist.")
    print("Es werden keine API-Seiten erneut geladen.")

else:
    # Wenn bereits Daten vorhanden sind, wird angehängt.
    # Beim ersten Start wird die Datei neu erstellt.
    file_mode = "a" if documents_saved > 0 else "w"

    try:
        with JSONL_FILE.open(
            file_mode,
            encoding="utf-8",
        ) as output_file:

            while next_url is not None:

                # Optionaler Testabbruch
                if (
                    MAX_PAGES is not None
                    and pages_downloaded >= MAX_PAGES
                ):
                    print()
                    print(
                        f"Testgrenze erreicht: MAX_PAGES = {MAX_PAGES}"
                    )
                    break

                page_number = pages_downloaded + 1

                print(
                    f"Lade Seite {page_number} ...",
                    end=" ",
                    flush=True,
                )

                try:
                    data = request_json(next_url)

                except (
                    requests.RequestException,
                    ValueError,
                    TypeError,
                    KeyError,
                ) as error:

                    error_message = (
                        f"Fehler auf Seite {page_number}; "
                        f"URL: {next_url}; "
                        f"{type(error).__name__}: {error}"
                    )

                    print("FEHLER")
                    print(error_message)

                    log_error(error_message)

                    print()
                    print(
                        "Der bisherige Fortschritt bleibt gespeichert."
                    )
                    print(
                        "Führe dieselbe Zelle später erneut aus, "
                        "um fortzufahren."
                    )

                    break

                documents = data.get("objects", [])
                meta = data.get("meta", {})

                number_saved = write_documents(
                    documents,
                    output_file,
                )

                documents_saved += number_saved
                pages_downloaded += 1

                next_url = meta.get("next")
                finished = next_url is None

                save_checkpoint(
                    next_url=next_url,
                    pages_downloaded=pages_downloaded,
                    documents_saved=documents_saved,
                    finished=finished,
                )

                print(
                    f"OK – {number_saved} Dokumente; "
                    f"gesamt {documents_saved}"
                )

                if pages_downloaded == 1:
                    total_count = meta.get("total_count")

                    print()
                    print("Gesamtzahl laut API:", total_count)
                    print()

                if pages_downloaded % REPORT_EVERY_PAGES == 0:
                    print("-" * 65)
                    print(
                        f"Zwischenstand: {pages_downloaded} Seiten, "
                        f"{documents_saved} Dokumente"
                    )
                    print("-" * 65)

                if finished:
                    print()
                    print("Die letzte API-Seite wurde erreicht.")
                    break

                time.sleep(PAUSE_SECONDS)

    except KeyboardInterrupt:
        print()
        print()
        print("Download wurde manuell unterbrochen.")
        print("Der bisherige Fortschritt bleibt gespeichert.")
        print(
            "Führe dieselbe Zelle erneut aus, um weiterzumachen."
        )

    finally:
        session.close()


# ============================================================
# 5. ERGEBNIS PRÜFEN
# ============================================================

print()
print("=" * 65)
print("AKTUELLER STAND")
print("=" * 65)
print("Geladene Seiten:      ", pages_downloaded)
print("Gespeicherte Dokumente:", documents_saved)
print("JSONL-Datei vorhanden:", JSONL_FILE.exists())

if JSONL_FILE.exists():
    jsonl_size_mb = JSONL_FILE.stat().st_size / (1024 ** 2)
    print(
        "JSONL-Dateigröße:     ",
        f"{jsonl_size_mb:.2f} MB",
    )


# ============================================================
# 6. CSV ERZEUGEN
# ============================================================

current_checkpoint = load_checkpoint()
download_finished = bool(
    current_checkpoint.get("finished", False)
)

if download_finished and JSONL_FILE.exists():

    print()
    print("Alle API-Seiten wurden geladen.")
    print("Erzeuge jetzt die vollständige CSV-Datei ...")

    try:
        df = pd.read_json(
            JSONL_FILE,
            lines=True,
        )

        # Eventuelle doppelte Dokumente anhand der ID entfernen
        if "id" in df.columns:
            rows_before = len(df)

            df = (
                df
                .drop_duplicates(
                    subset=["id"],
                    keep="last",
                )
                .reset_index(drop=True)
            )

            duplicates_removed = rows_before - len(df)

        else:
            duplicates_removed = 0

        # Dateigröße zusätzlich in MB
        if "file_size" in df.columns:
            df["file_size_mb"] = (
                pd.to_numeric(
                    df["file_size"],
                    errors="coerce",
                )
                / (1024 ** 2)
            ).round(2)

        df.to_csv(
            CSV_FILE,
            index=False,
            encoding="utf-8-sig",
        )

        print()
        print("CSV erfolgreich erstellt.")
        print("DataFrame-Größe:", df.shape)
        print(
            "Entfernte Duplikate:",
            duplicates_removed,
        )
        print("CSV-Datei:", CSV_FILE)

        print()
        print("Erste fünf Dokumente:")
        display(df.head())

    except Exception as error:
        print()
        print("Die JSONL-Datei wurde gespeichert,")
        print("aber die CSV-Erzeugung ist fehlgeschlagen.")
        print(type(error).__name__, error)

else:
    print()
    print("Der Download ist noch nicht vollständig abgeschlossen.")
    print(
        "Die CSV-Datei wird erst nach der letzten API-Seite erzeugt."
    )
    print(
        "Du kannst die Zelle jederzeit erneut ausführen, "
        "um fortzufahren."
    )

FRAGDENSTAAT-METADATEN-DOWNLOAD
Arbeitsordner:        c:\Users\Admin\Desktop\OpenLens\Datenbank
JSONL-Datei:          c:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_alle_dokumente.jsonl
CSV-Datei:            c:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_alle_dokumente.csv
Checkpoint-Datei:     c:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_checkpoint.json
Bereits geladene Seiten: 5164
Bereits gespeicherte Dokumente: 258184

Der Checkpoint meldet, dass der Download bereits abgeschlossen ist.
Es werden keine API-Seiten erneut geladen.

AKTUELLER STAND
Geladene Seiten:       5164
Gespeicherte Dokumente: 258184
JSONL-Datei vorhanden: True
JSONL-Dateigröße:      410.95 MB

Alle API-Seiten wurden geladen.
Erzeuge jetzt die vollständige CSV-Datei ...

CSV erfolgreich erstellt.
DataFrame-Größe: (258184, 26)
Entfernte Duplikate: 0
CSV-Datei: c:\Users\Admin\Desktop\OpenLens\Datenbank\fragdenstaat_alle_dokumente.csv

Erste fünf Dokumente:


,resource_uri,id,site_url,title,slug,description,published_at,num_pages,public,listed,...,outline,properties,uid,data,pages_uri,original,foirequest,publicbody,last_modified_at,file_size_mb
0,https://fragdenstaat.de/api/v1/document/26/,26,https://fragdenstaat.de/dokumente/26-abgewiese...,Abgewiesene Anträge auf Einstweilige Verfügung...,abgewiesene-antrage-auf-einstweilige-verfugung...,,NaN,33,True,True,...,,{'_format_webp': True},65db47b4-4d6a-4ebe-accf-cf9992e14716,{},/api/v1/page/?document=26,NaN,NaN,NaN,2018-06-27 20:05:07.319893+00:00,2.55
1,https://fragdenstaat.de/api/v1/document/27/,27,https://fragdenstaat.de/dokumente/27-unsere-kl...,Unsere Klage gegen das BMI,unsere-klage-gegen-das-bmi,,NaN,29,True,True,...,,{'_format_webp': True},eb31ebd7-f018-426a-8aed-551973d5d39e,{},/api/v1/page/?document=27,NaN,NaN,NaN,2018-06-27 20:10:16.622665+00:00,2.03
2,https://fragdenstaat.de/api/v1/document/28/,28,https://fragdenstaat.de/dokumente/28-anerkennu...,Anerkennung Ansprüche durch BMI,anerkennung-ansprueche-durch-bmi,,NaN,4,True,True,...,,{'_format_webp': True},272201b6-e432-452b-b590-7b40ede42a06,{},/api/v1/page/?document=28,NaN,NaN,NaN,2018-06-27 20:13:07.669626+00:00,0.41
3,https://fragdenstaat.de/api/v1/document/43/,43,https://fragdenstaat.de/dokumente/43-klage-geg...,Klage gegen das Innenministerium: Twitter-Dire...,klage-gegen-das-innenministerium-twitter-direk...,,NaN,27,True,True,...,,{'_format_webp': True},15411a80-48ff-4fb8-9acb-6d0f501138d6,{},/api/v1/page/?document=43,https://fragdenstaat.de/api/v1/attachment/42203/,https://fragdenstaat.de/api/v1/request/29951/,NaN,2018-09-24 09:16:00.252193+00:00,0.82
4,https://fragdenstaat.de/api/v1/document/35/,35,https://fragdenstaat.de/dokumente/35/,bka-barschel-vermerk.pdf,,,NaN,7,True,True,...,,{'_format_webp': True},fc4cbe19-bbbe-460a-aaef-b94c8b3018c5,{},/api/v1/page/?document=35,https://fragdenstaat.de/api/v1/attachment/40784/,https://fragdenstaat.de/api/v1/request/31847/,https://fragdenstaat.de/api/v1/publicbody/82/,2018-08-07 10:10:28.323197+00:00,9.29
